# Monet-Style Image Generation with DCGAN  
## Deep Learning Specialization — Course 5, Week 5 (Peer-graded Assignment)

In this project, I train a Generative Adversarial Network (GAN) to synthesize new Monet-style images using the Kaggle “GAN – Getting Started” dataset. The goal is to build a well-structured generative model, analyze its behavior during training, and evaluate the realism of generated samples using Kaggle’s MiFID score.


## 1. Problem Description and Dataset

The task is to teach a neural network to create new images that resemble Monet’s paintings.  
Unlike classification or regression, here the model’s goal is to *generate* data from scratch.

The dataset contains Monet-style images, typically sized (insert after EDA). These images exhibit distinctive textures, color palettes, and painterly style. Our model must learn these characteristics well enough to produce new images that could plausibly belong to the dataset.

Kaggle evaluates the generated images using **MiFID**, an improved variant of Fréchet Inception Distance that takes both realism and memorization into account.



## 2. Imports and Setup

In this section, I load the main libraries, set a few global constants, and define paths to the Kaggle dataset. This makes the rest of the notebook easier to read and modify.



In [2]:
# -----------------------------
# Paths to the dataset (local or Kaggle)
# -----------------------------

# Local path to your dataset folder
local_path = Path("G:/IDL Module 5/Module-5-IDL/data")

# Kaggle path
kaggle_path = Path("/kaggle/input/gan-getting-started")

# Automatically detect environment
if kaggle_path.exists():
    DATA_ROOT = kaggle_path
    print("Running on Kaggle.")
elif local_path.exists():
    DATA_ROOT = local_path
    print("Running locally.")
else:
    raise FileNotFoundError("Dataset not found in local directory or Kaggle input directory.")

# Folders inside the dataset
MONET_DIR = DATA_ROOT / "monet_jpg"
PHOTO_DIR = DATA_ROOT / "photo_jpg"

print("Monet directory exists:", MONET_DIR.exists())
print("Photo directory exists:", PHOTO_DIR.exists())


NameError: name 'Path' is not defined

## 3. Exploratory Data Analysis (EDA)

Before building the model, we inspect the dataset to understand:
- how many Monet images are available,
- their resolution and aspect ratio,
- the overall color distribution and style,
- whether preprocessing steps such as resizing or normalization are required.

Visualizing sample images helps form intuition about what the generator is expected to learn.


## 4. Data Preprocessing

For training stability, all images are resized to a fixed resolution (e.g., 64×64) and scaled to the range \([-1, 1]\).  
GANs with a final `tanh` activation expect inputs in this range, which leads to smoother gradients and more stable convergence.

We create a `tf.data.Dataset` pipeline that efficiently loads, shuffles, batches, and prefetches data during training.


## 5. Theoretical Background

### 5.1 From Autoencoders to Generative Models
Autoencoders compress images into low-dimensional latent codes and reconstruct them back.  
Variational Autoencoders (VAEs) extend this by learning a *probabilistic* latent space. While VAEs can generate new data, the images often appear overly smooth.

### 5.2 Generative Adversarial Networks (GANs)
GANs take a different approach: two networks compete with each other in a game-like training process.

- **Generator (G):** starts from a random latent vector \( z \), sampled from a simple distribution like a Gaussian. This vector acts as a hidden “blueprint” that the network transforms into an image.
- **Discriminator (D):** tries to distinguish real Monet paintings from generated ones.

Their objectives push each other to improve until the generator produces convincing samples.

### 5.3 DCGAN Architecture
The DCGAN architecture uses:
- transposed convolutions to gradually build up an image from a latent vector,
- Batch Normalization to stabilize learning,
- ReLU activations in the generator (except for the final `tanh` layer),
- LeakyReLU in the discriminator,
- strided convolutions instead of pooling.

These design decisions were shown to produce stable and detailed image generation.

### 5.4 Improved Training Techniques
From the “Improved GAN Training” paper, useful ideas include:
- **Feature matching:** guiding the generator to match statistics of intermediate discriminator layers.
- **Minibatch discrimination:** helping the discriminator detect when the generator collapses to similar images.
- **One-sided label smoothing:** reducing overconfidence of D and making gradients smoother.

In this project, the primary focus is on building a stable DCGAN, with an option to extend it using feature matching if desired.


## 6. Generator Architecture

The generator begins with a simple latent vector \( z \), sampled from a normal distribution.  
It uses fully connected and transposed convolutional layers to gradually transform this vector into a full-color Monet-like image. Batch Normalization helps maintain healthy gradient flow, while `tanh` produces final pixel values in the expected range \([-1, 1]\).


## 7. Discriminator Architecture

The discriminator receives an image and tries to determine whether it is real or generated.  
It uses a stack of strided convolutional layers with LeakyReLU activations to detect the texture patterns and color structures characteristic of Monet paintings. The final output is a single probability score.


## 8. Loss Functions and Optimization

We use the standard binary cross-entropy loss:

- The discriminator learns to assign high scores to real images and low scores to generated ones.
- The generator learns to produce images that persuade the discriminator to label them as real.

To stabilize training, we use the non-saturating generator loss:

\[
L_G = - \mathbb{E}[\log D(G(z))].
\]

Both networks are trained using the Adam optimizer with parameters recommended from the DCGAN paper:  
**learning rate = 2e-4**, **beta1 = 0.5**.


## 9. Training Procedure

Training proceeds in alternating steps:

1. Sample a batch of real Monet images.  
2. Sample a batch of latent vectors \( z \) and generate fake images.  
3. Update the discriminator using both real and fake batches.  
4. Update the generator to make its images more convincing.  

Throughout training, we periodically save generated samples to visualize how the model evolves. This helps diagnose issues like mode collapse or training instability.


## 10. Results: Training Dynamics and Generated Images

We inspect:

- loss curves for both networks,
- sample images produced at regular intervals,
- visual progression of the generator’s capabilities.

Early images often appear noisy or abstract, but over time the generator learns structure, color palettes, and recognizable patterns characteristic of Monet’s style.


## 11. Generating Submission Images

Once the model is trained, we generate 10,000 Monet-style images for Kaggle submission:

- sample new latent vectors \( z \),
- create images using the final generator,
- rescale them back to \([0, 255]\),
- save each image into a folder with the required naming format.

These images are then zipped and uploaded to Kaggle for MiFID evaluation.


## 12. Discussion and Conclusion

In this project, I implemented a DCGAN to synthesize Monet-style images.  
The model successfully captured many of the color patterns and textures present in the training data. Training was sensitive to hyperparameters, but the DCGAN architecture provided stable results once properly configured.

Potential improvements include:
- experimenting with feature matching or minibatch discrimination,
- training on higher resolution images,
- exploring conditional or style-transfer models such as CycleGAN.

Finally, I submitted 10,000 generated samples to Kaggle and obtained a MiFID score of **[insert score]**, demonstrating that the model produces visually coherent Monet-like images.


## 13. References

- I. Goodfellow et al., *“Generative Adversarial Networks”* (2014).  
- A. Radford et al., *“DCGAN: Unsupervised Representation Learning with Deep Convolutional GANs”* (2015).  
- T. Salimans et al., *“Improved Techniques for Training GANs”* (2016).  
- D. Kingma and M. Welling, *“Auto-Encoding Variational Bayes”* (2013).  
- Keras Blog: *“Building Autoencoders in Keras”*.  
- Kaggle Competition: *“GAN — Getting Started”*.
